# Training Notebook - Random Forest Pipeline

This notebook sets up a scikit-learn pipeline mirroring the EDA preprocessing workflow. It trains a `RandomForestClassifier` model, evaluates its capabilities, and logs metrics/models to MLflow.

In [6]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_recall_curve, PrecisionRecallDisplay
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from utility.main import read_data

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'imblearn'

In [ ]:
def load_and_initial_clean():
    df = read_data()
    # Replace ? with nan
    df = df.replace('?', np.nan)
    # Drop duplicates
    df = df.drop_duplicates()
    
    # Drop columns with high missingness
    cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    # Drop rows missing critical values
    df = df.dropna(subset=['race', 'diag_1', 'diag_2', 'diag_3', 'gender'])
    
    # Drop all null columns
    all_null_cols = df.columns[df.isnull().all()]
    df = df.drop(columns=all_null_cols, errors='ignore')
    
    # Remove Unknown/Invalid gender
    df = df[df['gender'] != 'Unknown/Invalid']
    
    # Create target variable (1 if readmitted <30 days)
    df['target'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
    
    # Drop IDs and original target to prevent data leakage
    df = df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted'], errors='ignore')
    
    return df

df = load_and_initial_clean()
print(f"Data shape after initial cleaning: {df.shape}")

In [ ]:
class CustomMapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        
        # 1. Age Mapping
        age_map = {
            '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, 
            '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, 
            '[80-90)': 8, '[90-100)': 9
        }
        if 'age' in X_copy.columns:
            X_copy['age'] = X_copy['age'].replace(age_map)
            
        # 2. Binary Mappings
        if 'change' in X_copy.columns:
            X_copy['change'] = X_copy['change'].replace({'No': 0, 'Ch': 1}).astype(float)
        if 'diabetesMed' in X_copy.columns:
            X_copy['diabetesMed'] = X_copy['diabetesMed'].replace({'No': 0, 'Yes': 1}).astype(float)
        if 'gender' in X_copy.columns:
            X_copy['gender'] = X_copy['gender'].replace({'Male': 0, 'Female': 1}).astype(float)
            
        # 3. Constant columns dropping + near zero variance
        constant_cols = ['citoglipton', 'metformin-rosiglitazone', 'examide', 'acetohexamide', 'troglitazone', 'glimepiride-pioglitazone']
        X_copy = X_copy.drop(columns=constant_cols, errors='ignore')
                
        return X_copy

In [ ]:
# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Grouping columns for specific transformations
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses'
]
diag_cols = ['diag_1', 'diag_2', 'diag_3']
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 
    'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 
    'rosiglitazone', 'acarbose', 'miglitol', 'tolazamide', 
    'insulin', 'glyburide-metformin', 'glipizide-metformin', 
    'metformin-pioglitazone'
]
constant_cols = ['citoglipton', 'metformin-rosiglitazone', 'examide', 'acetohexamide', 'troglitazone', 'glimepiride-pioglitazone']
active_medication_cols = [col for col in medication_cols if col not in constant_cols]
# Add medication columns to cat_cols so they go through OneHotEncoder
cat_cols = ['race', 'max_glu_serum', 'A1Cresult'] + active_medication_cols

numerical_transformer = SimpleImputer(strategy='mean')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Target Encoder handles diagnoses columns
# Note: smoothing mitigates data-leakage / overfitting
target_encoder_transformer = TargetEncoder(smooth='auto')

col_transformer = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
        ('diag_target_enc', target_encoder_transformer, diag_cols)
    ],
    remainder='passthrough'  # Keep transformed age, gender from CustomMapper
)

preprocessor = Pipeline(steps=[
    ('custom_mapper', CustomMapper()),
    ('col_transform', col_transformer)
])

# Preview pipeline components
preprocessor

In [ ]:
# Setup MLflow Tracking
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Diabetic_Readmission")

with mlflow.start_run(run_name="Random_Forest_Pipeline"):
    # Define Model - Fix: use class_weight="balanced"
    model = RandomForestClassifier(
        n_estimators=200, 
        max_depth=12, 
        min_samples_leaf=5, 
        class_weight="balanced", 
        random_state=42
    )
    
    # Full classification pipeline using ImbPipeline to support SMOTE
    # Note: Flattened to avoid TypeError: All intermediate steps should not be Pipelines
    full_pipeline = ImbPipeline(steps=[
        ("custom_mapper", CustomMapper()),
        ("col_transform", col_transformer),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])
    
    print("Training Pipeline Configured. Fitting model...")
    full_pipeline.fit(X_train, y_train)
    
    print("Evaluating Model...")
    y_proba = full_pipeline.predict_proba(X_test)[:, 1]
    
    # Threshold tuning
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_threshold = thresholds[np.argmax(f1_scores[:-1])]
    print(f"Optimal Threshold: {best_threshold:.4f}")
    
    y_pred = (y_proba >= best_threshold).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    f1_binary = f1_score(y_test, y_pred)
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    f1_macro = f1_score(y_test, y_pred, average="macro")
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score (Binary): {f1_binary:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    # Logging to MLflow
    mlflow.log_param("model_type", "Random Forest with SMOTE")
    mlflow.log_params(model.get_params())
    mlflow.log_param("optimal_threshold", best_threshold)
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score_binary", f1_binary)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Log the complete pipeline
    mlflow.sklearn.log_model(full_pipeline, "model", serialization_format="cloudpickle")
    
    print("Logging to MLflow complete!")


In [ ]:
def plot_curves(estimator, X, y, metric="f1"):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=3, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 5), scoring=metric
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_mean, "o-", color="r", label="Training score")
    plt.plot(train_sizes, test_mean, "o-", color="g", label="Cross-validation score")
    plt.title(f"Learning Curve ({metric.capitalize()})")
    plt.xlabel("Training Examples")
    plt.ylabel("Score")
    plt.legend(loc="best")
    plt.grid()

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plot_curves(full_pipeline, X_train, y_train, metric="f1")

plt.subplot(1, 2, 2)
plot_curves(full_pipeline, X_train, y_train, metric="accuracy")
plt.tight_layout()
plt.show()

# 2. Precision-Recall Curve
plt.figure(figsize=(8, 6))
PrecisionRecallDisplay.from_estimator(full_pipeline, X_test, y_test, ax=plt.gca())
plt.title("Precision-Recall Curve (Test Set)")
plt.grid()
plt.show()